<a href="https://colab.research.google.com/github/deepjal-28/Anganwadi-Childcare-Centre-Mapping/blob/main/Anganwadi_Childcare_Centre_Mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install libraries
!pip install geopandas folium scikit-learn pyproj mapclassify -q

# Upload your zip files
from google.colab import files
uploaded = files.upload()

# Extract all zips
import zipfile, os
for zip_name in uploaded.keys():
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall("data/")

print("✅ Files extracted:")
for f in os.listdir("data/"):
    print(" →", f)

Saving ward_boundary.zip to ward_boundary (1).zip
Saving Bhuvaneshwar_roads.zip to Bhuvaneshwar_roads (2).zip
Saving anganwadi_centres_shapefile.zip to anganwadi_centres_shapefile (2).zip
✅ Files extracted:
 → ward_boundary
 → anganwadi_centres_shapefile
 → Bhuvaneshwar_roads


In [8]:
import geopandas as gpd
import pandas as pd
import os

# Helper to find any shapefile in a directory tree
def find_shapefile(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.shp'):
                return os.path.join(root, file)
    return None

# Locate the shapefiles dynamically
ward_path = find_shapefile("data/ward_boundary")
anganwadi_path = find_shapefile("data/anganwadi_centres_shapefile")

print("📍 Found Ward Shapefile:", ward_path)
print("📍 Found Anganwadi Shapefile:", anganwadi_path)

# Load the shapefiles using the dynamically found paths
ward = gpd.read_file(ward_path)
anganwadi = gpd.read_file(anganwadi_path)

# Check what columns exist in your ward file
print("=== WARD BOUNDARY COLUMNS ===")
print(ward.columns.tolist())
print("\n=== FIRST FEW ROWS ===")
print(ward.head())

print("\n=== ANGANWADI COLUMNS ===")
print(anganwadi.columns.tolist())
print(anganwadi.head())

📍 Found Ward Shapefile: data/ward_boundary/ward_boundary.shp
📍 Found Anganwadi Shapefile: data/anganwadi_centres_shapefile/anganwadi_centres.shp
=== WARD BOUNDARY COLUMNS ===
['objectid', 'area_in_he', 'typei', 'branchesi', 'coordinate', 'xi', 'yi', 'longitudei', 'latitudei', 'lengthi', 'areai', 'bearingi', 'selectionm', 'selectioni', 'versioni', 'sno', 'wardno', 'municipalz', 'nameofthec', 'mobilenoof', 'numberofho', 'totalwardp', 'totalmalep', 'totalfemal', 'totalscpop', 'scmalepopu', 'scfemalepo', 'totalstpop', 'stmalepopu', 'stfemalepo', 'WardLevelO', 'shape.STLe', 'WardLeve_1', 'geometry']

=== FIRST FEW ROWS ===
   objectid  area_in_he  typei  branchesi  coordinate             xi  \
0         1    325.4890      3          1          75  378118.285432   
1         2    120.1880      3          1          74  375563.922050   
2         3    139.1290      3          1          81  375602.800683   
3         4     97.2715      3          1          35  379153.924827   
4         5   

In [9]:
# After running Step 2, check your column names
# Then rename them to standard names for the rest of the code

# ⚠️ CHANGE these names to match YOUR shapefile columns
ward = ward.rename(columns={
    "WARD_NO"   : "ward_no",      # ward number column
    "HH"        : "households",   # household count
    "TOT_P"     : "total_pop",    # total population
    "TOT_M"     : "male_pop",     # male population
    "TOT_F"     : "female_pop",   # female population
})

# Verify renaming worked
print(ward[["ward_no", "households", "total_pop", "male_pop", "female_pop"]].head(10))
print("\nTotal wards:", len(ward))  # should show 67

  ward_no  households  total_pop  male_pop  female_pop
0      W9        3328      13932      7302        6630
1     W25        3207      13508      7068        6440
2     W51        2904      12138      6519        5619
3     W56        2855      11228      5853        5375
4      W6        1964      13547      6757        6790
5      W7        3420      13698      7188        6510
6     W13        3516      13287      7002        6285
7     W27        2958      12039      6293        5746
8     W12        3227      13243      7454        5789
9     W11        2930      11834      6235        5599

Total wards: 67


In [10]:
# Reproject all to UTM Zone 44N (meters) for distance calculations
ward      = ward.to_crs(epsg=32644)
anganwadi = anganwadi.to_crs(epsg=32644)

print("✅ CRS aligned:", ward.crs)

✅ CRS aligned: EPSG:32644


In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Normalize population and household data
ward["pop_norm"]  = scaler.fit_transform(ward[["total_pop"]])
ward["hh_norm"]   = scaler.fit_transform(ward[["households"]])

# Female ratio (higher = more women = more anganwadi need)
ward["female_ratio"] = ward["female_pop"] / ward["total_pop"]
ward["female_norm"]  = scaler.fit_transform(ward[["female_ratio"]])

# Combined Need Score
ward["need_score"] = (
    ward["pop_norm"]    * 0.40 +
    ward["hh_norm"]     * 0.35 +
    ward["female_norm"] * 0.25
)

print("✅ Need Score calculated!")
print(ward[["ward_no", "total_pop", "households", "need_score"]]
      .sort_values("need_score", ascending=False).head(10))

✅ Need Score calculated!
   ward_no  total_pop  households  need_score
42      W4      16185        3476    0.970345
62     W60      15459        3310    0.868458
18     W35      14130        3422    0.770764
28     W19      14071        3517    0.763088
0       W9      13932        3328    0.749653
5       W7      13698        3420    0.741389
61      W8      13529        3575    0.735726
13     W24      13636        3331    0.730853
63     W61      13522        3269    0.727098
32     W43      13773        3241    0.721392


In [12]:
# Count how many anganwadis fall inside each ward
anganwadi_in_ward = gpd.sjoin(anganwadi, ward[["ward_no", "geometry"]],
                               how="left", predicate="within")

count_per_ward = anganwadi_in_ward.groupby("ward_no").size().reset_index(name="anganwadi_count")

# Merge count back to ward
ward = ward.merge(count_per_ward, on="ward_no", how="left")
ward["anganwadi_count"] = ward["anganwadi_count"].fillna(0)

# Anganwadi per 1000 people
ward["anganwadi_per_1000"] = (ward["anganwadi_count"] / ward["total_pop"]) * 1000

print("✅ Coverage calculated!")
print(ward[["ward_no", "total_pop", "anganwadi_count", "anganwadi_per_1000"]]
      .sort_values("anganwadi_count"))

✅ Coverage calculated!
   ward_no  total_pop  anganwadi_count  anganwadi_per_1000
5       W7      13698              0.0            0.000000
30     W33      12453              0.0            0.000000
47      W1      12378              0.0            0.000000
13     W24      13636              1.0            0.073335
23     W37      11679              1.0            0.085624
..     ...        ...              ...                 ...
37     W41      13525             13.0            0.961183
43      W5      13637             13.0            0.953289
44     W10      13540             13.0            0.960118
49     W22      13568             13.0            0.958137
35     W20      11677             17.0            1.455853

[67 rows x 4 columns]


In [13]:
# Coverage Score (inverse — fewer anganwadis = higher gap)
ward["coverage_norm"] = scaler.fit_transform(ward[["anganwadi_count"]])
ward["gap_score"] = (
    ward["need_score"]    * 0.60 +
    (1 - ward["coverage_norm"]) * 0.40
)

# Classify wards
ward["priority"] = pd.cut(ward["gap_score"],
                           bins=3,
                           labels=["🟢 Low Priority", "🟡 Medium Priority", "🔴 High Priority"])

print("\n📊 Priority Distribution:")
print(ward["priority"].value_counts())

print("\n🔴 TOP 10 HIGH PRIORITY WARDS:")
print(ward[["ward_no", "total_pop", "anganwadi_count", "gap_score", "priority"]]
      .sort_values("gap_score", ascending=False).head(10))


📊 Priority Distribution:
priority
🟡 Medium Priority    46
🔴 High Priority      16
🟢 Low Priority        5
Name: count, dtype: int64

🔴 TOP 10 HIGH PRIORITY WARDS:
   ward_no  total_pop  anganwadi_count  gap_score         priority
42      W4      16185              5.0   0.864560  🔴 High Priority
62     W60      15459              3.0   0.850487  🔴 High Priority
5       W7      13698              0.0   0.844834  🔴 High Priority
13     W24      13636              1.0   0.814982  🔴 High Priority
28     W19      14071              2.0   0.810794  🔴 High Priority
61      W8      13529              3.0   0.770847  🔴 High Priority
6      W13      13287              3.0   0.759769  🔴 High Priority
25     W39      12716              1.0   0.753392  🔴 High Priority
4       W6      13547              1.0   0.746718  🔴 High Priority
18     W35      14130              5.0   0.744811  🔴 High Priority


In [15]:
from sklearn.cluster import KMeans
import numpy as np

# Focus on high gap wards
high_gap_wards = ward[ward["gap_score"] > ward["gap_score"].quantile(0.6)].copy()
centroids = high_gap_wards.geometry.centroid

coords  = np.array([[pt.x, pt.y] for pt in centroids])
weights = high_gap_wards["gap_score"].values

# Suggest 5 new anganwadi locations
N = 5
kmeans = KMeans(n_clusters=N, random_state=42)
kmeans.fit(coords, sample_weight=weights)

suggested = kmeans.cluster_centers_
print(f"✅ {N} new anganwadi locations suggested!")

✅ 5 new anganwadi locations suggested!


In [16]:
import folium
from pyproj import Transformer

# Convert to WGS84 for mapping
ward_map      = ward.to_crs(epsg=4326)
anganwadi_map = anganwadi.to_crs(epsg=4326)

# Base map
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="CartoDB positron")

# Choropleth — Gap Score by Ward
folium.Choropleth(
    geo_data=ward_map.__geo_interface__,
    data=ward_map,
    columns=["ward_no", "gap_score"],
    key_on="feature.properties.ward_no",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Gap Score (Red = Needs More Care Centers)",
    nan_fill_color="white"
).add_to(m)

# Ward labels
for _, row in ward_map.iterrows():
    centroid = row.geometry.centroid
    folium.Tooltip(
        f"Ward {row['ward_no']} | Pop: {int(row['total_pop'])} | Anganwadi: {int(row['anganwadi_count'])}"
    )

# Existing Anganwadis — Blue dots
for _, row in anganwadi_map.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color="blue",
        fill=True,
        fill_opacity=0.8,
        popup="🏠 Existing Anganwadi"
    ).add_to(m)

# Suggested New Locations — Red stars
transformer = Transformer.from_crs("EPSG:32644", "EPSG:4326", always_xy=True)
for i, loc in enumerate(suggested):
    lon, lat = transformer.transform(loc[0], loc[1])
    folium.Marker(
        location=[lat, lon],
        popup=f"⭐ Suggested New Anganwadi {i+1}",
        icon=folium.Icon(color="red", icon="star")
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:10px; border-radius:8px; border:1px solid grey;">
  <b>Legend</b><br>
  🔵 Existing Anganwadi<br>
  ⭐ Suggested New Location<br>
  🟥 High Gap (High Priority)<br>
  🟨 Medium Gap<br>
  🟩 Low Gap
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("bhubaneswar_anganwadi_map.html")
print("✅ MAP SAVED → open bhubaneswar_anganwadi_map.html")

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


✅ MAP SAVED → open bhubaneswar_anganwadi_map.html


In [17]:
from IPython.display import display, HTML

# Display map inline in Colab
with open("bhubaneswar_anganwadi_map.html", "r") as f:
    map_html = f.read()

display(HTML(map_html))

In [18]:
from google.colab import files

# Download the HTML file to your PC
files.download("bhubaneswar_anganwadi_map.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# Change this line in your map code:

# ❌ OLD (needs API key)
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="CartoDB positron")

# ✅ NEW OPTIONS (completely free, no API key needed)

# Option 1 — OpenStreetMap (default, most detailed)
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="OpenStreetMap")

# Option 2 — Clean light map
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="cartodbpositron")

# Option 3 — Satellite view
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="Esri.WorldImagery")

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [20]:
import folium
from IPython.display import display, HTML
from pyproj import Transformer

# Recreate map with FREE tiles
m = folium.Map(location=[20.27, 85.84], zoom_start=12, tiles="OpenStreetMap")

# Choropleth
folium.Choropleth(
    geo_data=ward_map.__geo_interface__,
    data=ward_map,
    columns=["ward_no", "gap_score"],
    key_on="feature.properties.ward_no",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Gap Score (Red = Needs More Care Centers)",
    nan_fill_color="white"
).add_to(m)

# Existing Anganwadis — Blue dots
for _, row in anganwadi_map.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color="blue", fill=True, fill_opacity=0.8,
        popup="🏠 Existing Anganwadi"
    ).add_to(m)

# Suggested New Locations — Red stars
transformer = Transformer.from_crs("EPSG:32644", "EPSG:4326", always_xy=True)
for i, loc in enumerate(suggested):
    lon, lat = transformer.transform(loc[0], loc[1])
    folium.Marker(
        location=[lat, lon],
        popup=f"⭐ Suggested New Anganwadi {i+1}",
        icon=folium.Icon(color="red", icon="star")
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:10px; border-radius:8px; border:1px solid grey; font-size:13px;">
  <b>📍 Legend</b><br>
  🔵 Existing Anganwadi<br>
  ⭐ Suggested New Location<br>
  🟥 High Gap Zone<br>
  🟨 Medium Gap Zone<br>
  🟩 Low Gap Zone
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save
m.save("bhubaneswar_anganwadi_map.html")

# ✅ Display directly in Colab
display(HTML(open("bhubaneswar_anganwadi_map.html").read()))